# 6.16 · 异常检测 / Anomaly Detection

> **课程定位 / Where this fits**
> 无监督的一个核心任务: 找"**不正常**"的点——欺诈、入侵、设备故障、数据错误。和 5.14 不平衡分类不同, 异常检测常常**没有标签**(或正例极少), 靠"异常=偏离多数模式"来识别。本课覆盖三大主流方法 + 基于重构(AE 6.15)的思路: **Isolation Forest / One-Class SVM / LOF**。
> Finding the "abnormal" points (fraud, intrusion, faults) — usually unsupervised. Covers Isolation Forest, One-Class SVM, LOF, plus reconstruction-based detection.

> 💡 **面试相关 / Interview-relevant**
> - "异常检测三大范式(统计/距离-密度/隔离)" ★★★★★
> - "Isolation Forest 原理(异常更易被隔离)" ★★★★★
> - "One-Class SVM 在做什么" ★★★★
> - "LOF 的局部密度思想 / 为何能抓局部异常" ★★★★
> - "novelty vs outlier detection 区别" ★★★★
> - "contamination 参数" ★★★

---

## 学习目标 / Learning Objectives
1. 异常检测范式 + novelty vs outlier。
2. **Isolation Forest**: 随机切, 异常路径短。
3. **One-Class SVM**: 学一个包住正常数据的边界。
4. **LOF**: 局部密度对比, 抓局部异常。
5. 重构误差(AE)做异常 + 方法对比/评估。

## 目录 / TOC
1. [范式与三大方法 ⭐](#1)
2. [🛰️ 数据: 网络入侵(合成)](#2)
3. [三方法对比 ⭐](#3)
4. [LOF: 局部异常 ⭐](#4)
5. [评估 + 重构误差思路](#5)
6. [小结](#6)


<a id="1"></a>
## 1. 范式与三大方法 ⭐ / Paradigms & Methods

**三类范式**:
- **统计/模型**: 拟合正常数据分布(高斯/GMM 6.6), 低概率=异常。
- **距离/密度**: 远离邻居 / 处于低密度区(LOF, DBSCAN 噪声点)。
- **隔离/集成**: Isolation Forest——异常更"孤立", 更容易被随机切分隔离。

**三大方法**:
| 方法 | 思想 | 适合 |
|---|---|---|
| **Isolation Forest** | 随机选特征随机切, 异常点平均**更少几刀**就被隔离 → 路径短 | 高维、大数据、首选 |
| **One-Class SVM** | 学一个把正常数据**包住**的边界(原点 vs 数据), 外面=异常 | 中小数据、非线性边界 |
| **LOF(局部离群因子)** | 比较每点与邻居的**局部密度**, 显著更稀疏=异常 | **局部**异常(不同密度区) |

**novelty vs outlier**(面试常问):
- **outlier detection**: 训练集里就含异常, 无监督找出它们(IsoForest/LOF 默认)。
- **novelty detection**: 训练集**全是正常**, 判断**新来的**点是否异常(One-Class SVM、LOF `novelty=True`)。


<a id="2"></a>
## 2. 数据: 网络入侵(合成) / Synthetic Network Intrusion

真 KDD Cup 99 入侵检测集很大。这里**内联合成**一个 KDD 风格数据: 大量"正常"网络连接(在几个特征上聚成一团)+ 少量"攻击"连接(偏离正常模式)。特征如连接时长、字节数、错误率等。约 4% 异常。


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")

def make_intrusion(n=2000, anomaly_rate=0.04, seed=0):
    rng = np.random.default_rng(seed)
    n_anom = int(n*anomaly_rate); n_norm = n - n_anom
    # 正常连接: 紧凑团
    normal = rng.multivariate_normal(
        [0.3, 0.5, 0.1], [[0.02,0.005,0],[0.005,0.03,0],[0,0,0.01]], n_norm)
    # 攻击: 多种偏离模式(高字节/高错误率/异常时长)
    a1 = rng.multivariate_normal([0.3,0.5,0.8], np.eye(3)*0.02, n_anom//2)   # 高错误率
    a2 = rng.multivariate_normal([1.5,2.0,0.1], np.eye(3)*0.1, n_anom-n_anom//2) # 高时长高字节
    X = np.vstack([normal, a1, a2])
    yt = np.r_[np.zeros(n_norm), np.ones(n_anom)].astype(int)
    idx = rng.permutation(len(X))
    return pd.DataFrame(X[idx], columns=["duration","src_bytes","error_rate"]), yt[idx]

X, ytrue = make_intrusion()
print(f"网络入侵(合成): {X.shape}, 异常率 {ytrue.mean():.1%} (1=攻击)")
fig, ax = plt.subplots(figsize=(6.5,5))
ax.scatter(X["duration"][ytrue==0], X["src_bytes"][ytrue==0], s=8, alpha=0.4, label="正常")
ax.scatter(X["duration"][ytrue==1], X["src_bytes"][ytrue==1], s=25, c="red", marker="x", label="攻击")
ax.set_xlabel("duration"); ax.set_ylabel("src_bytes"); ax.legend()
ax.set_title("网络连接: 正常聚成团, 攻击偏离(待无监督检测)")
plt.tight_layout(); plt.show()


<a id="3"></a>
## 3. 三方法对比 ⭐ / Three Methods Compared

`contamination` = 预期异常比例, 决定判定阈值。三者都用它。


In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, classification_report

Xs = StandardScaler().fit_transform(X)
cont = 0.04
methods = {
    "IsolationForest": IsolationForest(contamination=cont, random_state=0),
    "OneClassSVM": OneClassSVM(nu=cont, gamma="scale"),
    "LOF": LocalOutlierFactor(contamination=cont, n_neighbors=20),
}
preds = {}
for name, m in methods.items():
    if name == "LOF":
        p = m.fit_predict(Xs)                 # LOF: fit_predict (outlier 模式)
        score = -m.negative_outlier_factor_   # 越大越异常
    else:
        p = m.fit(Xs).predict(Xs)
        score = -m.score_samples(Xs)          # 越大越异常
    p = (p == -1).astype(int)                 # sklearn: -1=异常 → 1
    preds[name] = p
    print(f"{name:<16} ROC-AUC {roc_auc_score(ytrue, score):.3f}  "
          f"(检出 {p.sum()} 个异常, 真实 {ytrue.sum()} 个)")
print("\nIsolationForest/OneClassSVM 几乎完美; LOF 这里很差(<0.5)——")
print("因为攻击点扎堆成 40 个一组的小簇, 在'局部'看彼此密度并不低 → LOF 误判为正常。")
print("教训: LOF 假设异常是相对邻居孤立的; 异常成簇时 LOF 会失效, 此时用全局方法(IsoForest)。")


In [ ]:
# 可视化三方法的判定 / visualize detections
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, (name, p) in zip(axes, preds.items()):
    ax.scatter(X["duration"][p==0], X["src_bytes"][p==0], s=8, alpha=0.3, label="判正常")
    ax.scatter(X["duration"][p==1], X["src_bytes"][p==1], s=30, c="red", marker="x", label="判异常")
    ax.set_title(name); ax.set_xlabel("duration"); ax.legend()
axes[0].set_ylabel("src_bytes")
plt.tight_layout(); plt.show()
print("IsolationForest/OneClassSVM 准确标出偏离正常团的攻击; LOF 因攻击成簇而漏判(见上)")
print("IsolationForest 通常最稳健、最快, 是高维大数据首选")


<a id="4"></a>
## 4. LOF: 局部异常 ⭐ / Local Outliers

LOF 的独到之处: 抓**局部**异常。一个点也许在全局看不算远, 但相对**它自己的邻居**密度低很多, 就是局部异常。Isolation Forest/One-Class SVM 偏全局, LOF 在**多密度**场景更敏锐。


In [ ]:
# 一个稠密簇 + 一个稀疏簇, 各放一个"局部异常" / local outlier demo
rng = np.random.default_rng(1)
dense = rng.normal([0,0], 0.3, (200,2))
sparse = rng.normal([5,5], 1.2, (200,2))
local_out = np.array([[1.2, 1.2], [5, 9]])   # 第1个在稠密簇边缘(局部异常), 第2个远离稀疏簇
Xl = np.vstack([dense, sparse, local_out])

lof = LocalOutlierFactor(n_neighbors=20, contamination=0.02)
pl = (lof.fit_predict(Xl) == -1)
fig, ax = plt.subplots(figsize=(7,5.5))
ax.scatter(Xl[~pl,0], Xl[~pl,1], s=12, alpha=0.4, label="正常")
ax.scatter(Xl[pl,0], Xl[pl,1], s=80, facecolor="none", edgecolor="red", linewidth=2, label="LOF 判异常")
ax.set_title("LOF: 按'相对邻居的局部密度'判异常 → 能抓稠密簇边缘的局部离群点")
ax.legend(); plt.tight_layout(); plt.show()
print("LOF 比较每点与其邻居的局部密度比值, 故能发现'相对周围偏稀疏'的局部异常")


<a id="5"></a>
## 5. 评估 + 重构误差思路 / Evaluation & Reconstruction

- **评估**: 若有少量标签, 用 **ROC-AUC / PR-AUC**(5.1)对异常分数评估(异常是极不平衡的正类, PR-AUC 尤其相关)。完全无标签时只能靠业务复核。
- **重构误差法**(接 6.15 AE / 6.8 PCA): 在正常数据上训 PCA/自编码器, **重构误差大的点 = 异常**(模型没见过这种模式, 重构不好)。


In [ ]:
from sklearn.decomposition import PCA
from sklearn.metrics import average_precision_score
# PCA 重构误差做异常分数 / PCA reconstruction error as anomaly score
pca = PCA(n_components=2).fit(Xs)
recon = pca.inverse_transform(pca.transform(Xs))
recon_err = ((Xs - recon)**2).sum(1)        # 重构误差越大越异常
print(f"PCA 重构误差法  ROC-AUC {roc_auc_score(ytrue, recon_err):.3f}, PR-AUC {average_precision_score(ytrue, recon_err):.3f}")
print(f"IsolationForest ROC-AUC {roc_auc_score(ytrue, -IsolationForest(random_state=0).fit(Xs).score_samples(Xs)):.3f}")
print("\n不平衡场景看 PR-AUC(5.1); 重构误差法简单有效, AE(6.15)是其非线性升级版")


<a id="6"></a>
## 6. 小结 / Summary

```
异常检测(多为无监督): 统计(低概率) / 密度(LOF) / 隔离(IsolationForest)
IsolationForest: 随机切, 异常更少刀被隔离(路径短) → 高维大数据首选, 快
One-Class SVM: 学包住正常数据的边界, 外=异常; nu≈异常比例
LOF: 局部密度比, 抓"相对邻居偏稀疏"的局部异常; 多密度场景强
重构误差(PCA/AE): 正常数据训练, 重构差=异常
novelty(训练全正常,判新点) vs outlier(训练含异常,找出来); contamination=预期异常比例
评估: 有标签用 ROC-AUC/PR-AUC(不平衡看 PR-AUC)
```

### 💡 面试速查
1. **三范式**: 统计/密度/隔离; 三方法 IsolationForest / OneClassSVM / LOF
2. **Isolation Forest**: 异常更易被随机切隔离(路径短), 快、高维稳健、首选
3. **LOF**: 局部密度对比, 抓局部异常(全局方法漏掉的)
4. **novelty vs outlier**; **contamination** 设异常比例阈值
5. **重构误差**(PCA/AE)做异常; 评估用 PR-AUC(极不平衡)

### 下一节
**6.17 关联规则**——购物篮分析: "买了啤酒的人也常买尿布"。Apriori/FP-Growth 挖掘频繁项集, support/confidence/lift 度量规则强度。
